# JAL-RAKSHAK — Deep Learning Training (Local: VS Code / Antigravity)

Trains two classifiers with a Keras (TensorFlow) MLP instead of Random Forest / XGBoost / CatBoost:

- **Filter Health** — `HEALTHY` / `MODERATE` / `UNHEALTHY` (18 approved features, `Filter_Health_Score_pct` excluded — target leakage)
- **Water Quality** — `SAFE` / `WARNING` / `CRITICAL` (measured raw + treated water parameters, plus optional efficiency features)

Train/test split uses the dataset's existing `ML_Split` column only — no random re-splitting.

## Before you run this
1. Put this notebook and your dataset file (`JAL_RAKSHAK_8000_Final_FilterHealth_WaterQuality_Dataset.csv`) in the **same folder**.
2. In VS Code / Antigravity, install the **Jupyter** extension if you haven't already (search "Jupyter" in the Extensions panel).
3. Create and activate a virtual environment, then run the install cell below — or run `pip install -r requirements.txt` in a terminal first.
4. Open this file, pick your venv's Python as the notebook kernel (top-right kernel picker), then run cells top to bottom.


## 1. Install packages

Run once. If you already ran `pip install -r requirements.txt` in a terminal, you can skip this cell.

In [1]:

import sys
import tensorflow as tf

print("Python:", sys.version)
print("Python executable:", sys.executable)
print("TensorFlow:", tf.__version__)


Python: 3.12.7 (tags/v3.12.7:0b05ead, Oct  1 2024, 03:06:41) [MSC v.1941 64 bit (AMD64)]
Python executable: c:\Users\kunal\Downloads\DL\venv-dl\Scripts\python.exe
TensorFlow: 2.21.0


## 2. Imports

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    classification_report, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.inspection import permutation_importance
from sklearn.base import BaseEstimator, ClassifierMixin

from tensorflow import keras
from tensorflow.keras import layers, callbacks

import joblib

RANDOM_SEED = 42
tf.random.set_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

os.makedirs("models", exist_ok=True)
os.makedirs("outputs", exist_ok=True)


## 3. Dataset filename

Edit this if your file is named differently or is `.xlsx` instead of `.csv`.

In [ ]:
DATASET_FILENAME = "JAL_RAKSHAK_8000_Final_FilterHealth_WaterQuality_Dataset_60_40.csv"

assert os.path.exists(DATASET_FILENAME), (
    f"Could not find '{DATASET_FILENAME}' in the current folder: {os.getcwd()}\n"
    "Make sure the dataset file is in the same folder as this notebook."
)


AssertionError: Could not find 'JAL_RAKSHAK_8000_Final_FilterHealth_WaterQuality_Dataset.csv' in the current folder: c:\Users\kunal\Downloads\DL
Make sure the dataset file is in the same folder as this notebook.

## 4. Config — exact feature lists and leakage rules from the spec

Do not add columns here without checking the "DO NOT USE AS ML FEATURES" section of the original requirements first.

In [ ]:
# ---------------------------------------------------------------------------
# MODEL 1 — FILTER HEALTH PREDICTION
# ---------------------------------------------------------------------------
FILTER_HEALTH_TARGET = "Filter_Health"
FILTER_HEALTH_CLASSES = ["HEALTHY", "MODERATE", "UNHEALTHY"]  # fixed label order

FILTER_HEALTH_FEATURES = [
    # Actual hardware measurements
    "Raw_TDS_ppm", "Raw_Turbidity_NTU", "Raw_Temperature_C",
    "Treated_TDS_ppm", "Treated_Turbidity_NTU", "Treated_pH", "Treated_Temperature_C",
    "Flow_Rate_L_min",
    # Backend-derived features
    "TDS_Removal_Efficiency_pct", "Turbidity_Removal_Efficiency_pct", "Temperature_Difference_C",
    "Previous_TDS_Efficiency_pct", "Efficiency_Improvement_pp", "Rolling_3Cycle_Avg_Efficiency_pct",
    "Efficiency_Trend_pct", "Previous_Turbidity_Efficiency_pct",
    "Rolling_3Cycle_Avg_Turbidity_Efficiency_pct", "Cycle_Number",
]
assert len(FILTER_HEALTH_FEATURES) == 18, "Filter Health feature list must have exactly 18 entries."

# Present in the dataset for engineering validation ONLY — never a model input (target leakage).
LEAKAGE_COLUMN_FILTER_HEALTH = "Filter_Health_Score_pct"

FILTER_HEALTH_WATCHLIST = [
    "Flow_Rate_L_min", "Turbidity_Removal_Efficiency_pct", "Previous_Turbidity_Efficiency_pct",
    "Rolling_3Cycle_Avg_Turbidity_Efficiency_pct", "Efficiency_Improvement_pp", "TDS_Removal_Efficiency_pct",
]

# ---------------------------------------------------------------------------
# MODEL 2 — WATER QUALITY CLASSIFICATION
# ---------------------------------------------------------------------------
WATER_QUALITY_TARGET = "Water_Quality_Class"
WATER_QUALITY_CLASSES = ["SAFE", "WARNING", "CRITICAL"]  # fixed label order

WATER_QUALITY_PRIMARY_FEATURES = [
    "Raw_TDS_ppm", "Raw_Turbidity_NTU", "Raw_Temperature_C",
    "Treated_TDS_ppm", "Treated_Turbidity_NTU", "Treated_pH", "Treated_Temperature_C",
]

# Spec: "You may additionally evaluate" these — included by default.
WATER_QUALITY_EXTRA_FEATURES = [
    "TDS_Removal_Efficiency_pct", "Turbidity_Removal_Efficiency_pct", "Temperature_Difference_C",
]
WATER_QUALITY_FEATURES = WATER_QUALITY_PRIMARY_FEATURES + WATER_QUALITY_EXTRA_FEATURES

# ---------------------------------------------------------------------------
# NEVER USE AS ML FEATURES (either task)
# ---------------------------------------------------------------------------
NEVER_USE_AS_FEATURES = ["Sample_ID", "Water_Quality_Class", "Filter_Health", "ML_Split", "Filter_Health_Score_pct"]

# TRAIN/TEST split — spec requires using the EXISTING ML_Split column only.
SPLIT_COLUMN = "ML_Split"
TRAIN_VALUE = "TRAIN"
TEST_VALUE = "TEST"


## 5. Load and inspect the dataset

In [ ]:
def load_dataset(path):
    if path.lower().endswith(".csv"):
        df = pd.read_csv(path)
    else:
        df = pd.read_excel(path)
    print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
    return df


def inspect_data(df):
    print("\n--- DATA INSPECTION ---")
    print(f"Shape: {df.shape}")
    print("\nColumn dtypes:")
    print(df.dtypes)

    missing = df.isnull().sum()
    missing = missing[missing > 0]
    print("\nNo missing values found." if len(missing) == 0 else f"\nMissing values:\n{missing}")

    print(f"\nDuplicate rows: {df.duplicated().sum()}")

    if SPLIT_COLUMN in df.columns:
        print(f"\n{SPLIT_COLUMN} value counts:")
        print(df[SPLIT_COLUMN].value_counts())


df = load_dataset(DATASET_FILENAME)
inspect_data(df)


## 6. Train/test split — using the existing `ML_Split` column only

In [ ]:
def split_by_ml_split(df):
    if SPLIT_COLUMN not in df.columns:
        raise ValueError(f"Expected column '{SPLIT_COLUMN}' not found in dataset.")
    train_df = df[df[SPLIT_COLUMN] == TRAIN_VALUE].copy()
    test_df = df[df[SPLIT_COLUMN] == TEST_VALUE].copy()
    if len(train_df) == 0 or len(test_df) == 0:
        raise ValueError(f"Split produced an empty set (train={len(train_df)}, test={len(test_df)}).")
    print(f"Train rows: {len(train_df)} | Test rows: {len(test_df)}")
    return train_df, test_df


def prepare_features_targets(train_df, test_df, feature_cols, target_col, class_order):
    missing_cols = [c for c in feature_cols if c not in train_df.columns]
    if missing_cols:
        raise ValueError(f"Missing expected feature columns: {missing_cols}")

    X_train = train_df[feature_cols].copy()
    X_test = test_df[feature_cols].copy()
    y_train_raw = train_df[target_col].copy()
    y_test_raw = test_df[target_col].copy()

    label_encoder = LabelEncoder()
    label_encoder.fit(class_order)  # fixed order, not alphabetical/data-driven
    y_train = label_encoder.transform(y_train_raw)
    y_test = label_encoder.transform(y_test_raw)

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    return X_train_scaled, X_test_scaled, y_train, y_test, scaler, label_encoder


train_df, test_df = split_by_ml_split(df)


## 7. Deep learning model — build / train / evaluate / feature importance

In [ ]:
def build_mlp(input_dim, num_classes):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),

        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.2),

        layers.Dense(32, activation="relu"),
        layers.Dropout(0.1),

        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def train_model(model, X_train, y_train, task_name, epochs=150, batch_size=32, val_split=0.15):
    checkpoint_path = f"models/{task_name}_dl_model.keras"

    class_weights_arr = compute_class_weight(class_weight="balanced", classes=np.unique(y_train), y=y_train)
    class_weight_dict = {i: w for i, w in enumerate(class_weights_arr)}
    print(f"Class weights for {task_name}: {class_weight_dict}")

    cb_list = [
        callbacks.EarlyStopping(monitor="val_loss", patience=15, restore_best_weights=True),
        callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=7, min_lr=1e-6),
        callbacks.ModelCheckpoint(checkpoint_path, monitor="val_loss", save_best_only=True),
    ]

    history = model.fit(
        X_train, y_train,
        validation_split=val_split,
        epochs=epochs,
        batch_size=batch_size,
        class_weight=class_weight_dict,
        callbacks=cb_list,
        verbose=2,
    )
    return model, history


def plot_training_history(history, task_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history["loss"], label="train_loss")
    axes[0].plot(history.history["val_loss"], label="val_loss")
    axes[0].set_title(f"{task_name} - Loss"); axes[0].set_xlabel("Epoch"); axes[0].legend()

    axes[1].plot(history.history["accuracy"], label="train_acc")
    axes[1].plot(history.history["val_accuracy"], label="val_acc")
    axes[1].set_title(f"{task_name} - Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].legend()

    plt.tight_layout()
    plt.savefig(f"outputs/{task_name}_training_curves.png", dpi=150)
    plt.show()


def evaluate_model(model, X_test, y_test, class_names, task_name):
    y_prob = model.predict(X_test, verbose=0)
    y_pred = np.argmax(y_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_test, y_pred, average="macro", zero_division=0)
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_test, y_pred, average="weighted", zero_division=0)

    report = classification_report(y_test, y_pred, target_names=class_names, zero_division=0, output_dict=True)
    cm = confusion_matrix(y_test, y_pred)

    metrics = {
        "task": task_name, "accuracy": acc,
        "precision_macro": precision_macro, "recall_macro": recall_macro, "f1_macro": f1_macro,
        "precision_weighted": precision_weighted, "recall_weighted": recall_weighted, "f1_weighted": f1_weighted,
        "classification_report": report, "confusion_matrix": cm.tolist(), "class_order": class_names,
    }

    print(f"\n=== {task_name} — Test Set Evaluation ===")
    print(f"Accuracy:        {acc:.4f}")
    print(f"Macro Precision: {precision_macro:.4f}")
    print(f"Macro Recall:    {recall_macro:.4f}")
    print(f"Macro F1:        {f1_macro:.4f}")
    print(f"Weighted F1:     {f1_weighted:.4f}")
    print(classification_report(y_test, y_pred, target_names=class_names, zero_division=0))

    with open(f"outputs/{task_name}_metrics.json", "w") as f:
        json.dump(metrics, f, indent=2)

    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
    plt.title(f"{task_name} - Confusion Matrix"); plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.tight_layout()
    plt.savefig(f"outputs/{task_name}_confusion_matrix.png", dpi=150)
    plt.show()

    return metrics, y_pred, y_prob


class KerasClassifierWrapper(BaseEstimator, ClassifierMixin):
    """sklearn-compatible wrapper so permutation_importance works on a Keras model
    (neural nets have no built-in .feature_importances_ like tree models do)."""
    def __init__(self, keras_model):
        self.keras_model = keras_model
        self.classes_ = None

    def fit(self, X, y):
        self.classes_ = np.unique(y)
        return self

    def predict(self, X):
        probs = self.keras_model.predict(X, verbose=0)
        return np.argmax(probs, axis=1)


def compute_feature_importance(model, X_test, y_test, feature_names, task_name, n_repeats=10):
    wrapper = KerasClassifierWrapper(model)
    wrapper.classes_ = np.unique(y_test)

    result = permutation_importance(
        wrapper, X_test, y_test, scoring="f1_macro", n_repeats=n_repeats, random_state=RANDOM_SEED)

    importance_df = pd.DataFrame({
        "feature": feature_names,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }).sort_values("importance_mean", ascending=False)

    importance_df.to_csv(f"outputs/{task_name}_feature_importance.csv", index=False)
    print(importance_df.to_string(index=False))

    plt.figure(figsize=(8, max(4, len(feature_names) * 0.35)))
    plt.barh(importance_df["feature"], importance_df["importance_mean"], xerr=importance_df["importance_std"])
    plt.gca().invert_yaxis()
    plt.xlabel("Permutation importance (drop in macro F1)")
    plt.title(f"{task_name} - Feature Importance (Permutation, DL model)")
    plt.tight_layout()
    plt.savefig(f"outputs/{task_name}_feature_importance.png", dpi=150)
    plt.show()

    return importance_df


## 8. Train the Filter Health model

`Filter_Health_Score_pct` is deliberately excluded from the feature list below — using it would leak the answer, since it's built from the same flow + turbidity-efficiency numbers that define filter health.

In [ ]:
assert LEAKAGE_COLUMN_FILTER_HEALTH not in FILTER_HEALTH_FEATURES, "Leakage column must not be a feature!"

X_train_fh, X_test_fh, y_train_fh, y_test_fh, scaler_fh, label_encoder_fh = prepare_features_targets(
    train_df, test_df, FILTER_HEALTH_FEATURES, FILTER_HEALTH_TARGET, FILTER_HEALTH_CLASSES
)

model_fh = build_mlp(input_dim=X_train_fh.shape[1], num_classes=len(FILTER_HEALTH_CLASSES))
model_fh.summary()

model_fh, history_fh = train_model(model_fh, X_train_fh, y_train_fh, task_name="filter_health")
plot_training_history(history_fh, "filter_health")

fh_class_names = list(label_encoder_fh.classes_)
fh_metrics, fh_y_pred, fh_y_prob = evaluate_model(model_fh, X_test_fh, y_test_fh, fh_class_names, "filter_health")


In [ ]:
fh_importance_df = compute_feature_importance(
    model_fh, X_test_fh, y_test_fh, FILTER_HEALTH_FEATURES, "filter_health"
)


In [ ]:
joblib.dump(scaler_fh, "models/filter_health_scaler.joblib")
joblib.dump(label_encoder_fh, "models/filter_health_label_encoder.joblib")
with open("models/filter_health_features.json", "w") as f:
    json.dump(FILTER_HEALTH_FEATURES, f, indent=2)
print("Filter Health model, scaler, and encoder saved in models/")


## 9. Train the Water Quality model

In [ ]:
X_train_wq, X_test_wq, y_train_wq, y_test_wq, scaler_wq, label_encoder_wq = prepare_features_targets(
    train_df, test_df, WATER_QUALITY_FEATURES, WATER_QUALITY_TARGET, WATER_QUALITY_CLASSES
)

model_wq = build_mlp(input_dim=X_train_wq.shape[1], num_classes=len(WATER_QUALITY_CLASSES))
model_wq.summary()

model_wq, history_wq = train_model(model_wq, X_train_wq, y_train_wq, task_name="water_quality")
plot_training_history(history_wq, "water_quality")

wq_class_names = list(label_encoder_wq.classes_)
wq_metrics, wq_y_pred, wq_y_prob = evaluate_model(model_wq, X_test_wq, y_test_wq, wq_class_names, "water_quality")


In [ ]:
wq_importance_df = compute_feature_importance(
    model_wq, X_test_wq, y_test_wq, WATER_QUALITY_FEATURES, "water_quality"
)


In [ ]:
joblib.dump(scaler_wq, "models/water_quality_scaler.joblib")
joblib.dump(label_encoder_wq, "models/water_quality_label_encoder.joblib")
with open("models/water_quality_features.json", "w") as f:
    json.dump(WATER_QUALITY_FEATURES, f, indent=2)
print("Water Quality model, scaler, and encoder saved in models/")


## 10. Combined summary table

In [ ]:
summary_df = pd.DataFrame([
    {
        "Task": "filter_health",
        "Accuracy": round(fh_metrics["accuracy"], 4),
        "Precision (macro)": round(fh_metrics["precision_macro"], 4),
        "Recall (macro)": round(fh_metrics["recall_macro"], 4),
        "F1 (macro)": round(fh_metrics["f1_macro"], 4),
        "F1 (weighted)": round(fh_metrics["f1_weighted"], 4),
    },
    {
        "Task": "water_quality",
        "Accuracy": round(wq_metrics["accuracy"], 4),
        "Precision (macro)": round(wq_metrics["precision_macro"], 4),
        "Recall (macro)": round(wq_metrics["recall_macro"], 4),
        "F1 (macro)": round(wq_metrics["f1_macro"], 4),
        "F1 (weighted)": round(wq_metrics["f1_weighted"], 4),
    },
])
summary_df.to_csv("outputs/combined_dl_summary.csv", index=False)
summary_df


## 11. Try a prediction on a new reading

This mirrors what `predict.py` does for the FastAPI backend — load the saved artifacts and score one reading.

In [ ]:
def predict_filter_health(reading: dict):
    x = np.array([[reading[feat] for feat in FILTER_HEALTH_FEATURES]], dtype=float)
    x_scaled = scaler_fh.transform(x)
    probs = model_fh.predict(x_scaled, verbose=0)[0]
    pred_idx = int(np.argmax(probs))
    pred_label = label_encoder_fh.inverse_transform([pred_idx])[0]
    return {
        "predicted_class": pred_label,
        "confidence": float(probs[pred_idx]),
        "class_probabilities": {cls: float(p) for cls, p in zip(label_encoder_fh.classes_, probs)},
    }


example_reading = {
    "Raw_TDS_ppm": 450, "Raw_Turbidity_NTU": 12.5, "Raw_Temperature_C": 26.0,
    "Treated_TDS_ppm": 180, "Treated_Turbidity_NTU": 1.8, "Treated_pH": 7.1,
    "Treated_Temperature_C": 25.5, "Flow_Rate_L_min": 1.6,
    "TDS_Removal_Efficiency_pct": 60.0, "Turbidity_Removal_Efficiency_pct": 85.6,
    "Temperature_Difference_C": 0.5, "Previous_TDS_Efficiency_pct": 62.0,
    "Efficiency_Improvement_pp": -2.0, "Rolling_3Cycle_Avg_Efficiency_pct": 61.0,
    "Efficiency_Trend_pct": -1.0, "Previous_Turbidity_Efficiency_pct": 86.0,
    "Rolling_3Cycle_Avg_Turbidity_Efficiency_pct": 85.8, "Cycle_Number": 42,
}
predict_filter_health(example_reading)


---
**Reminder:** this is an 8,000-row *synthetic* prototype dataset — not real field or lab data. High test accuracy here does not guarantee equivalent real-world accuracy. Validate against real ESP32/prototype readings before trusting either model in production. Keep this ML layer separate from your existing rule-based OPTIMIZE / CONTINUE / HOLD / NO_ACTION decision engine.

**Where everything gets saved** (relative to wherever this notebook file lives):
- `models/` — trained `.keras` models, scalers, label encoders, feature-list JSONs
- `outputs/` — metrics JSON, confusion matrix PNGs, training curve PNGs, feature importance CSV/PNG, `combined_dl_summary.csv`
